# 09 - Link prediction (edge-level tasks)

So far: labels on **nodes** and on whole **graphs**. The third task level is the
**edge**: *will a link exist between two nodes?* This is the engine behind recommendation
(user->item), knowledge-graph completion (entity->entity), and social-network friend
suggestion.

The standard recipe is **encoder -> decoder**:

1. **Encoder**: a GNN maps every node to an embedding `z_i` from the *observed* edges.
2. **Decoder**: score a candidate pair by a simple function of their embeddings, e.g.
   the dot product `z_i . z_j`.
3. **Negative sampling**: real graphs only give *positive* edges, so we sample
   non-edges as negatives and train a binary classifier.

In [1]:
import os, sys, warnings, time
warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.abspath(".."))   # make utils/ importable from notebooks/

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
pd.set_option("display.precision", 3)
torch.manual_seed(0); np.random.seed(0)

from utils import graphs as G     # synthetic graph generators (known ground truth)
from utils import models as M     # the model zoo (MLP, GCN, SAGE, GAT, GIN, GPS, ...)
from utils import training as T    # train/eval loops + metrics
from utils import plotting as P    # graph drawing, curves, comparisons


## Split edges into message-passing vs supervision

We hold out 10% of the (undirected) edges as the **test** links to predict. The model
only ever *sees* the remaining 90% (the message-passing graph). Negative examples are
random non-edges, drawn with PyG's `negative_sampling`.

In [2]:
from torch_geometric.utils import to_undirected, negative_sampling

data, gt = G.make_sbm_homophily(n_per_block=250, n_blocks=3, homophily=0.9, avg_degree=20, seed=1)
ei = data.edge_index
und = ei[:, ei[0] < ei[1]]                          # unique undirected edges
perm = torch.randperm(und.size(1))
n_test = int(0.1 * und.size(1))
test_pos  = und[:, perm[:n_test]]
train_pos = und[:, perm[n_test:]]
train_msg = to_undirected(train_pos)                # message-passing graph (both directions)
print(f"{und.size(1)} edges -> {train_pos.size(1)} train / {test_pos.size(1)} test (held out)")

7512 edges -> 6761 train / 751 test (held out)


In [3]:
import torch.nn as nn, torch.nn.functional as Fnn
from torch_geometric.nn import SAGEConv

class Encoder(nn.Module):
    def __init__(self, ind, hid, out):
        super().__init__()
        self.c1 = SAGEConv(ind, hid); self.c2 = SAGEConv(hid, out)
    def forward(self, x, ei):
        return self.c2(Fnn.relu(self.c1(x, ei)), ei)

def decode(z, edge):                                # dot-product decoder
    return (z[edge[0]] * z[edge[1]]).sum(-1)

from sklearn.metrics import roc_auc_score
T.set_seed(0)
enc = Encoder(data.num_features, 64, 32)
opt = torch.optim.Adam(enc.parameters(), lr=0.01)

for epoch in range(101):
    enc.train(); opt.zero_grad()
    z = enc(data.x, train_msg)
    neg = negative_sampling(train_msg, num_nodes=data.num_nodes,
                            num_neg_samples=train_pos.size(1))
    logits = torch.cat([decode(z, train_pos), decode(z, neg)])
    labels = torch.cat([torch.ones(train_pos.size(1)), torch.zeros(neg.size(1))])
    loss = Fnn.binary_cross_entropy_with_logits(logits, labels)
    loss.backward(); opt.step()

# Evaluate on held-out positives vs fresh negatives.
enc.eval()
with torch.no_grad():
    z = enc(data.x, train_msg)
    test_neg = negative_sampling(und, num_nodes=data.num_nodes, num_neg_samples=test_pos.size(1))
    scores = torch.cat([decode(z, test_pos), decode(z, test_neg)]).sigmoid()
    labels = torch.cat([torch.ones(test_pos.size(1)), torch.zeros(test_neg.size(1))])
print(f"link-prediction test ROC-AUC = {roc_auc_score(labels, scores):.3f}")

link-prediction test ROC-AUC = 0.777


## Reading the result - and the ground-truth ceiling

The encoder learns embeddings where nodes in the **same community** sit close together,
so their dot product scores within-community pairs higher - and our SBM is homophilous,
so true links *are* mostly within communities. An AUC well above 0.5 means the model
recovered that prior.

But notice it lands around **0.8, not 1.0** - and that's *correct*, not a failure. We
planted the graph as a **stochastic** block model: given two nodes' communities, whether
they're actually connected is a **coin flip**. So there is genuinely *no* signal that
separates a connected same-community pair from an unconnected one beyond the community
prior. We know the generative ceiling, and the GNN gets close to it - exactly the kind of
check this series is built around. On real graphs where links are driven by features or
richer structure (citation, co-purchase, knowledge graphs), the same recipe reaches much
higher AUC.

In practice you'd also report **Hits@k** / **MRR** (rank each true edge against many
negatives) and use stronger decoders (MLP on `[z_i || z_j]`, or DistMult/RotatE for
knowledge graphs).

**When you're doing link prediction**

- (+) Recommendation, knowledge-graph completion, de-duplication, friend/co-author
  suggestion - anywhere the *relationships* are the thing you want to predict.
- The encoder is just a GNN - **everything from notebooks 02-08 applies** (pick GCN/SAGE
  for homophily, attention for noisy neighbourhoods, transformers for long range).
- (!) Negative sampling and evaluation protocol matter a lot; be careful not to leak
  test edges into the message-passing graph.

Finally: **notebook 10** ties it all together into a *prescription* - which model when,
and when to skip GNNs entirely.